---
## Cell 1 — Configuration & Imports

In [1]:
!pip install -q numpy pandas matplotlib seaborn shap scikit-learn lightgbm


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — edit only this block
# ═══════════════════════════════════════════════════════════════
DATASET_PATH = 'dataset_Disease.csv'
FIG_DIR      = './dissertation_figures'
ARTIFACT_DIR = './model_artifacts'
SHAP_N       = 150        # samples for SHAP — keep low for RAM
CV_FOLDS     = 5
RANDOM_STATE = 42
DPI          = 180
# ═══════════════════════════════════════════════════════════════

import gc, os, pickle, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import shap

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, confusion_matrix, roc_curve
)
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','lightgbm','-q'], check=True)
    from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
os.makedirs(FIG_DIR,      exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

PALETTE = ['#2E4057','#4472C4','#ED7D31','#70AD47','#FF4B4B']
plt.rcParams.update({
    'figure.dpi'       : DPI,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'savefig.dpi'      : DPI,
    'savefig.bbox'     : 'tight',
})

def savefig(name):
    path = f'{FIG_DIR}/{name}'
    plt.savefig(path, bbox_inches='tight')
    plt.close('all')
    gc.collect()
    print(f'  ✓  {name}')

print('✓ Cell 1 complete — imports & config ready')

d:\msc final report\dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Cell 1 complete — imports & config ready


---
## Cell 2 — Load Dataset

In [3]:
df = pd.read_csv(DATASET_PATH)
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(' ','_',regex=False)
    .str.replace(r'[^a-z0-9_]','',regex=True)
)
target_col = next((c for c in df.columns if c in ('disease', 'diseases','prognosis','label','target')), None)
if target_col is None:
    raise ValueError(f'No target column found. Columns: {list(df.columns[:10])}')
df.rename(columns={target_col:'Disease'}, inplace=True)
df['Disease'] = df['Disease'].str.strip()

SYMPTOM_COLS = [c for c in df.columns if c != 'Disease']
df[SYMPTOM_COLS] = df[SYMPTOM_COLS].fillna(0).astype(np.int8)

cc = df['Disease'].value_counts()
print(f'Rows       : {len(df):,}')
print(f'Diseases   : {df["Disease"].nunique()}')
print(f'Symptoms   : {len(SYMPTOM_COLS)}')
print(f'Missing    : {df.isnull().sum().sum()}')
print(f'Memory     : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(f'Balance    : min={cc.min()}, max={cc.max()}, mean={cc.mean():.0f}')
print('✓ Cell 2 complete — dataset loaded')

Rows       : 246,945
Diseases   : 773
Symptoms   : 377
Missing    : 0
Memory     : 104.5 MB
Balance    : min=1, max=1219, mean=319
✓ Cell 2 complete — dataset loaded


---
## Cell 3 — Figure 01: Class Distribution

In [4]:
counts      = df['Disease'].value_counts().sort_values(ascending=False)
N_CLS_TOTAL = len(counts)

fig, axes = plt.subplots(1, 2, figsize=(18, max(9, min(N_CLS_TOTAL//8, 16))))
fig.suptitle(
    f'Class Distribution Across {N_CLS_TOTAL} Disease Categories\n'
    f'(Total={len(df):,}  |  Min={counts.min()}  |  Max={counts.max()}  |  Mean={counts.mean():.0f})',
    fontsize=13, y=1.02
)
ax = axes[0]
ax.hist(counts.values, bins=min(50,N_CLS_TOTAL), color=PALETTE[1], edgecolor='white', alpha=0.88)
ax.axvline(counts.mean(),   color=PALETTE[0], linestyle='--', lw=1.8, label=f'Mean={counts.mean():.0f}')
ax.axvline(counts.median(), color=PALETTE[2], linestyle=':',  lw=1.8, label=f'Median={counts.median():.0f}')
ax.set_xlabel('Instances per Disease Class'); ax.set_ylabel('Number of Disease Classes')
ax.set_title('A: Distribution of Class Sizes'); ax.legend(fontsize=10)

ax2 = axes[1]
TOP_N, BOT_N = 20, 10
top_s = counts.head(TOP_N); bot_s = counts.tail(BOT_N)
sep   = f'… {N_CLS_TOTAL-TOP_N-BOT_N} classes not shown …'
labels= list(bot_s.index[::-1])+[sep]+list(top_s.index[::-1])
values= list(bot_s.values[::-1])+[0]+list(top_s.values[::-1])
colors_=[PALETTE[2]]*BOT_N+['#cccccc']+[PALETTE[1]]*TOP_N
bars  = ax2.barh(range(len(labels)), values, color=colors_, edgecolor='white', alpha=0.88, height=0.7)
ax2.set_yticks(range(len(labels))); ax2.set_yticklabels([str(l)[:50] for l in labels], fontsize=8.5)
for bar,val,lbl in zip(bars,values,labels):
    if lbl==sep or val==0: continue
    ax2.text(val+max(values)*0.01, bar.get_y()+bar.get_height()/2, str(int(val)), va='center', fontsize=8)
ax2.set_xlabel('Number of Instances')
ax2.set_title(f'B: Top {TOP_N} and Bottom {BOT_N} Disease Classes')
ax2.legend(handles=[
    mpatches.Patch(color=PALETTE[1],alpha=0.85,label=f'Top {TOP_N} (most common)'),
    mpatches.Patch(color=PALETTE[2],alpha=0.85,label=f'Bottom {BOT_N} (least common)'),
], fontsize=9, loc='lower right')
plt.tight_layout()
savefig('fig01_class_distribution.png')
print('✓ Cell 3 complete')

  ✓  fig01_class_distribution.png
✓ Cell 3 complete


---
## Cell 4 — Figure 02: Symptom Frequencies

In [5]:
symptom_freq = df[SYMPTOM_COLS].sum().sort_values(ascending=False)
TOP30_SYMS   = symptom_freq.head(30)
labels_clean = [s.replace('_',' ') for s in TOP30_SYMS.index]

fig, ax = plt.subplots(figsize=(13, 11))
colors  = plt.cm.viridis(np.linspace(0.85, 0.15, 30))
bars    = ax.barh(range(30), TOP30_SYMS.values[::-1], color=colors, edgecolor='white', height=0.72)
ax.set_yticks(range(30))
ax.set_yticklabels(labels_clean[::-1], fontsize=10)
for bar,val in zip(bars, TOP30_SYMS.values[::-1]):
    ax.text(val+TOP30_SYMS.max()*0.005, bar.get_y()+bar.get_height()/2,
            f'{int(val):,}', va='center', fontsize=8.5)
ax.set_xlabel('Total Occurrences Across Dataset')
ax.set_title('Top 30 Most Frequent Symptoms in the Diseases and Symptoms Dataset', pad=14)
ax.set_xlim(0, TOP30_SYMS.max()*1.12)
plt.tight_layout()
savefig('fig02_symptom_frequency.png')
print('✓ Cell 4 complete')

  ✓  fig02_symptom_frequency.png
✓ Cell 4 complete


---
## Cell 5 — Figure 03: Correlation Heatmap

In [6]:
top30_names = TOP30_SYMS.index.tolist()
corr        = df[top30_names].astype(np.float32).corr()
mask        = np.triu(np.ones_like(corr, dtype=bool))
labels_hm   = [s.replace('_',' ') for s in top30_names]

fig, ax = plt.subplots(figsize=(20, 17))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    xticklabels=labels_hm, yticklabels=labels_hm,
    linewidths=0.4, linecolor='white',
    cbar_kws={'label':'Pearson Correlation','shrink':0.7,'pad':0.02})
ax.xaxis.tick_top(); ax.xaxis.set_label_position('top')
ax.tick_params(axis='x', labelsize=10, rotation=40, pad=2)
ax.tick_params(axis='y', labelsize=10, rotation=0)
ax.set_title(
    'Symptom Co-occurrence Correlation Matrix (Top 30 Symptoms)\n'
    'Positive = symptoms frequently co-occur; Negative = mutually exclusive',
    pad=80, fontsize=13)
plt.tight_layout()
savefig('fig03_correlation_heatmap.png')
del corr; gc.collect()
print('✓ Cell 5 complete')

  ✓  fig03_correlation_heatmap.png
✓ Cell 5 complete


---
## Cell 6 — Figure 04: CRISP-DM Diagram

In [7]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0,16); ax.set_ylim(0,7); ax.axis('off')
phases = [
    ('Phase 1\nBusiness &\nClinical\nUnderstanding','#2E4057',0.4),
    ('Phase 2\nData\nUnderstanding','#4472C4',3.0),
    ('Phase 3\nData\nPreparation','#ED7D31',5.6),
    ('Phase 4\nModelling\nDT · RF · LR · LGB','#70AD47',8.2),
    ('Phase 5\nEvaluation\nAcc · F1 · MCC · AUC','#FF4B4B',10.8),
    ('Phase 6\nDeployment\nFlask + SHAP','#7030A0',13.4),
]
BW, BH = 2.3, 4.4
for label,color,x in phases:
    rect = mpatches.FancyBboxPatch((x,1.3),BW,BH,
        boxstyle='round,pad=0.15',facecolor=color,edgecolor='white',linewidth=1.8,alpha=0.93)
    ax.add_patch(rect)
    ax.text(x+BW/2,1.3+BH/2,label,ha='center',va='center',
            fontsize=10,color='white',fontweight='bold',linespacing=1.6)
for i in range(len(phases)-1):
    ax.annotate('',xy=(phases[i+1][2]+0.05,3.5),xytext=(phases[i][2]+BW+0.05,3.5),
        arrowprops=dict(arrowstyle='->',color='#555',lw=2.2))
ax.annotate('',xy=(phases[0][2]+BW/2,1.25),xytext=(phases[-1][2]+BW/2,1.25),
    arrowprops=dict(arrowstyle='->',color='#aaa',lw=1.5,connectionstyle='arc3,rad=-0.35'))
ax.text(8,0.35,'Iterative feedback loop',ha='center',fontsize=9.5,color='#888',style='italic')
ax.set_title('CRISP-DM Framework Applied to the Multi-Disease Prediction System',pad=14,fontsize=13)
plt.tight_layout()
savefig('fig04_crisp_dm.png')
print('✓ Cell 6 complete')

  ✓  fig04_crisp_dm.png
✓ Cell 6 complete


---
## Cell 7 — Data Preparation & Split

In [8]:
# Drop singleton classes — can't stratify-split with <2 samples
cc_raw  = df['Disease'].value_counts()
dropped = cc_raw[cc_raw < 2].index.tolist()
if dropped:
    print(f'⚠  Dropping {len(dropped)} singleton class(es)')
    df = df[~df['Disease'].isin(dropped)].copy()
    print(f'   Rows after drop: {len(df):,}')
else:
    print('✓ All classes have ≥2 samples')

le          = LabelEncoder()
y           = le.fit_transform(df['Disease'])
CLASS_NAMES = le.classes_
N_CLASSES   = len(CLASS_NAMES)
X           = df[SYMPTOM_COLS].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

# Cast to float32 immediately — never let float64 enter training
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)

_, tr_counts = np.unique(y_train, return_counts=True)
_, te_counts = np.unique(y_test,  return_counts=True)
print(f'Train: {X_train.shape}   Test: {X_test.shape}')
print(f'Classes: {N_CLASSES}')
print(f'Train balance — min/max: {tr_counts.min()}/{tr_counts.max()}')
print(f'Test  balance — min/max: {te_counts.min()}/{te_counts.max()}')

CV_FOLDS_SAFE = min(CV_FOLDS, int(tr_counts.min()))
if CV_FOLDS_SAFE < CV_FOLDS:
    print(f'\n⚠  CV_FOLDS reduced {CV_FOLDS} → {CV_FOLDS_SAFE} (min class={tr_counts.min()} samples)')
else:
    print(f'\n✓  CV_FOLDS={CV_FOLDS_SAFE} safe for all {N_CLASSES} classes')
CV_FOLDS = CV_FOLDS_SAFE
print('✓ Cell 7 complete')

⚠  Dropping 19 singleton class(es)
   Rows after drop: 246,926
Train: (197540, 377)   Test: (49386, 377)
Classes: 754
Train balance — min/max: 2/975
Test  balance — min/max: 1/244

⚠  CV_FOLDS reduced 5 → 2 (min class=2 samples)
✓ Cell 7 complete


---
## Cell 8 — Train All Four Models

In [9]:
# MODELS defined here so N_CLASSES is already known
MODELS = {
    'Random Forest': {
        'model': RandomForestClassifier(
            n_estimators=100, max_depth=20,
            min_samples_split=5, min_samples_leaf=3,
            max_features='sqrt', class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=1),
        'scaled': False,
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(
            max_depth=20, min_samples_split=10, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE),
        'scaled': False,
    },
    'Logistic Regression': {
        'model': LogisticRegression(
            C=1.0, solver='lbfgs', penalty='l2', max_iter=200, random_state=RANDOM_STATE),
        'scaled': True,
    },
    'LightGBM': {
        'model': LGBMClassifier(
            objective='multiclass', num_class=N_CLASSES,
            n_estimators=100, learning_rate=0.1,
            num_leaves=31, max_depth=8, min_child_samples=30,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, n_jobs=1, verbose=-1),
        'scaled': False,
    },
}

results  = {}
trained  = {}
y_probas = {}   # kept for ROC curves (Cell 12), freed after
scaler   = None # fitted in loop if LR is best model

for name, cfg in MODELS.items():
    print('\n' + '='*70)
    print(f'  TRAINING: {name}')
    print('='*70)

    # Scale on-demand — never hold raw + scaled in RAM simultaneously
    if cfg['scaled']:
        print('  Scaling...', end=' ', flush=True)
        _scaler = StandardScaler()
        Xtr = _scaler.fit_transform(X_train).astype(np.float32)
        Xte = _scaler.transform(X_test).astype(np.float32)
        scaler = _scaler
    else:
        Xtr = X_train
        Xte = X_test

    print('  Fitting...', end=' ', flush=True)
    model = cfg['model']
    model.fit(Xtr, y_train)
    print('done')

    y_pred = model.predict(Xte)

    # Standard classification metrics
    acc     = accuracy_score     (y_test, y_pred)
    prec    = precision_score    (y_test, y_pred, average='macro', zero_division=0)
    rec     = recall_score       (y_test, y_pred, average='macro', zero_division=0)
    f1      = f1_score           (y_test, y_pred, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    mcc     = matthews_corrcoef  (y_test, y_pred)
    kappa   = cohen_kappa_score  (y_test, y_pred)

    # AUC — sample 100 classes, normalize sliced probs, recode labels
    try:
        y_prob = model.predict_proba(Xte).astype(np.float32)
        unique_classes = np.unique(y_test)
        if len(unique_classes) > 100:
            rng_auc     = np.random.default_rng(RANDOM_STATE)
            sampled_cls = rng_auc.choice(unique_classes, size=100, replace=False)
        else:
            sampled_cls = unique_classes
        mask            = np.isin(y_test, sampled_cls)
        y_test_sub      = y_test[mask]
        y_prob_sub      = y_prob[mask][:, sampled_cls]
        row_sums        = y_prob_sub.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        y_prob_sub     /= row_sums
        mapping         = {c:i for i,c in enumerate(sampled_cls)}
        y_test_enc      = np.array([mapping[v] for v in y_test_sub], dtype=np.int32)
        auc = roc_auc_score(y_test_enc, y_prob_sub, multi_class='ovr', average='weighted')
        y_probas[name]  = y_prob   # keep full proba for ROC curves
        del y_prob_sub, y_test_sub, y_test_enc, mask, row_sums
        gc.collect()
    except Exception as e:
        print(f'  ⚠ AUC failed: {e}')
        auc = np.nan

    results[name] = {
        'Accuracy'         : round(acc,     4),
        'Balanced Acc'     : round(bal_acc, 4),
        'Precision'        : round(prec,    4),
        'Recall'           : round(rec,     4),
        'F1'               : round(f1,      4),
        'MCC'              : round(mcc,     4),
        'Kappa'            : round(kappa,   4),
        'AUC'              : round(float(auc),4) if not np.isnan(auc) else np.nan,
    }
    trained[name] = model

    print(f'  Acc={acc:.4f} | BalAcc={bal_acc:.4f} | F1={f1:.4f} | MCC={mcc:.4f} | Kappa={kappa:.4f} | AUC={auc:.4f}')

    del y_pred, model
    if cfg['scaled']:
        del Xtr, Xte, _scaler
    gc.collect()

results_df      = pd.DataFrame(results).T.sort_values('F1', ascending=False)
best_model_name = results_df.index[0]
print('\n' + '='*90)
print('MODEL PERFORMANCE SUMMARY')
print('='*90)
print(results_df.round(4).to_string())
print('='*90)
print(f'  ★  Best model (F1): {best_model_name}')
print('✓ Cell 8 complete')


  TRAINING: Random Forest
  Fitting... 

done
  Acc=0.5460 | BalAcc=0.6709 | F1=0.6082 | MCC=0.5528 | Kappa=0.5449 | AUC=0.9691

  TRAINING: Decision Tree
  Fitting... done
  ⚠ AUC failed: Target scores need to be probabilities for multiclass roc_auc, i.e. they should sum up to 1.0 over classes
  Acc=0.0301 | BalAcc=0.1043 | F1=0.0971 | MCC=0.0779 | Kappa=0.0267 | AUC=nan

  TRAINING: Logistic Regression
  Scaling...   Fitting... done
  Acc=0.8687 | BalAcc=0.8695 | F1=0.8430 | MCC=0.8683 | Kappa=0.8683 | AUC=0.9998

  TRAINING: LightGBM
  Fitting... done
  ⚠ AUC failed: Target scores need to be probabilities for multiclass roc_auc, i.e. they should sum up to 1.0 over classes
  Acc=0.0062 | BalAcc=0.0020 | F1=0.0008 | MCC=0.0083 | Kappa=0.0013 | AUC=nan

MODEL PERFORMANCE SUMMARY
                     Accuracy  Balanced Acc  Precision  Recall      F1     MCC   Kappa     AUC
Logistic Regression    0.8687        0.8695     0.8384  0.8672  0.8430  0.8683  0.8683  0.9998
Random Forest          0.5460        0.6709     0.6740  0.661

---
## Cell 9 — Figure 15: Cross-Validation Plot

In [10]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_scores = {}

for name, cfg in MODELS.items():
    if cfg['scaled']:
        _sc  = StandardScaler()
        Xuse = _sc.fit_transform(X_train).astype(np.float32)
        del _sc
    else:
        Xuse = X_train
    scores = cross_val_score(cfg['model'], Xuse, y_train,
                             cv=cv, scoring='f1_macro', n_jobs=1)
    cv_scores[name] = scores
    print(f'  {name:22s}: {scores.mean():.4f} ± {scores.std():.4f}')
    if cfg['scaled']: del Xuse
    gc.collect()

model_names = list(MODELS.keys())
n_folds     = CV_FOLDS

if n_folds >= 5:
    fig, ax = plt.subplots(figsize=(10, 6))
    bp = ax.boxplot([cv_scores[n] for n in model_names], labels=model_names,
        patch_artist=True, notch=False,
        medianprops=dict(color='white',linewidth=2.5),
        whiskerprops=dict(linewidth=1.3), capprops=dict(linewidth=1.3),
        flierprops=dict(marker='o',markersize=6,alpha=0.6))
    for patch,color in zip(bp['boxes'],PALETTE):
        patch.set_facecolor(color); patch.set_alpha(0.85)
    rng2 = np.random.default_rng(RANDOM_STATE)
    for i,(name,scores) in enumerate(cv_scores.items(),1):
        ax.scatter(rng2.normal(i,0.06,len(scores)),scores,s=30,color='white',alpha=0.8,zorder=3)
        ax.text(i,scores.mean()+0.01,f'{scores.mean():.3f}',ha='center',fontsize=9.5,fontweight='bold')
    ax.set_xticklabels(model_names,fontsize=11)
    ax.set_ylabel(f'Macro F1-Score ({n_folds}-fold CV)')
    ax.set_ylim(0, min(1.15, max(s.max() for s in cv_scores.values())*1.2))
else:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(model_names)); w = 0.8/n_folds
    fold_colors = PALETTE
    for fold_i in range(n_folds):
        fold_vals = [cv_scores[n][fold_i] for n in model_names]
        offset    = (fold_i - n_folds/2 + 0.5)*w
        bars      = ax.bar(x+offset, fold_vals, w*0.9, label=f'Fold {fold_i+1}',
                           color=fold_colors[fold_i%len(fold_colors)], alpha=0.82, edgecolor='white')
        for bar,v in zip(bars,fold_vals):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.008,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8, rotation=90)
    for i,name in enumerate(model_names):
        mv = cv_scores[name].mean()
        ax.hlines(mv,i-0.4,i+0.4,colors='black',lw=2.5,linestyles='--',zorder=5)
        ax.text(i,mv+0.025,f'μ={mv:.3f}',ha='center',fontsize=9.5,fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(model_names,fontsize=11)
    ax.set_ylabel(f'Macro F1-Score ({n_folds}-fold CV)')
    ax.legend(loc='upper right',fontsize=9,ncol=n_folds)
    ax.set_ylim(0, min(1.20, max(s.max() for s in cv_scores.values())*1.30))

ax.set_title(f'Cross-Validation Macro F1-Score — {n_folds}-Fold Stratified\n'
             'Dashed line = mean per model  |  Each point/bar = one fold', pad=12, fontsize=13)
plt.tight_layout()
savefig('fig15_cv_f1_boxplot.png')
print('✓ Cell 9 complete')

  Random Forest         : 0.5788 ± 0.0113
  Decision Tree         : 0.0844 ± 0.0033
  Logistic Regression   : 0.8470 ± 0.0019
  LightGBM              : 0.0003 ± 0.0001
  ✓  fig15_cv_f1_boxplot.png
✓ Cell 9 complete


---
## Cell 10 — Figures 05–08: Confusion Matrices

In [11]:
CM_SPECS = [
    (name, f'fig{5+i:02d}_{name.lower().replace(" ","_")}_confusion_matrix.png', 5+i)
    for i,name in enumerate(trained.keys())
]

for name, fname, fnum in CM_SPECS:
    if MODELS[name]['scaled']:
        _sc2 = StandardScaler()
        Xte  = _sc2.fit_transform(X_train).astype(np.float32)
        Xte  = _sc2.transform(X_test).astype(np.float32)
        del _sc2
    else:
        Xte = X_test

    y_pred      = trained[name].predict(Xte)
    test_labels = np.unique(y_test)
    cm          = confusion_matrix(y_test, y_pred, labels=test_labels)
    cm_pct      = cm.astype(float)/np.maximum(cm.sum(axis=1,keepdims=True),1)*100
    cm_cls      = le.classes_[test_labels]
    acc = results[name]['Accuracy']; f1 = results[name]['F1']

    if N_CLASSES <= 60:
        sz  = max(14, N_CLASSES*30/DPI)
        fig, ax = plt.subplots(figsize=(sz, sz*0.88))
        sns.heatmap(cm_pct, ax=ax, cmap='Blues', vmin=0, vmax=100,
            xticklabels=cm_cls, yticklabels=cm_cls,
            linewidths=0.4, linecolor='#e0e0e0',
            annot=(N_CLASSES<=25), fmt='.0f' if N_CLASSES<=25 else '',
            cbar_kws={'label':'Recall % per class','shrink':0.7})
        fs = max(5,min(9,280//N_CLASSES))
        plt.xticks(rotation=45,ha='right',fontsize=fs); plt.yticks(fontsize=fs)
        ax.set_xlabel('Predicted Class',labelpad=10); ax.set_ylabel('True Class',labelpad=10)
        fig.suptitle(f'{name} — Normalised Confusion Matrix\n'
            f'Accuracy={acc:.4f}  F1={f1:.4f} | {N_CLASSES} classes | n_test={len(y_test)}',
            fontsize=12, y=1.01)
    else:
        recall_per = np.diag(cm_pct)
        rs         = pd.Series(recall_per, index=cm_cls)
        TOP_N2, BOT_N2 = 25, 25
        top2 = rs.nlargest(TOP_N2); bot2 = rs.nsmallest(BOT_N2)
        bot2 = bot2[~bot2.index.isin(top2.index)]; BOT_N2 = len(bot2)
        sep2 = f'… {N_CLASSES-TOP_N2-BOT_N2} classes not shown …'
        cl2  = list(bot2.index[::-1])+[sep2]+list(top2.index[::-1])
        cv2  = list(bot2.values[::-1])+[np.nan]+list(top2.values[::-1])
        bc2  = [PALETTE[4]]*BOT_N2+['#cccccc']+[PALETTE[1]]*TOP_N2
        nb2  = len(cl2)
        fig  = plt.figure(figsize=(24, max(14, nb2*0.42+4)))
        gs   = fig.add_gridspec(1,2,width_ratios=[1,1.4],wspace=0.28)
        axH  = fig.add_subplot(gs[0])
        sns.heatmap(cm_pct,ax=axH,cmap='Blues',vmin=0,vmax=100,
            xticklabels=False,yticklabels=False,linewidths=0,rasterized=True,
            cbar_kws={'label':'Recall % per class','shrink':0.6})
        axH.set_xlabel('Predicted Class',fontsize=12); axH.set_ylabel('True Class',fontsize=12)
        axH.set_title(f'A: Full Confusion Matrix\n(each pixel = 1 of {N_CLASSES} classes)',fontsize=12)
        axB  = fig.add_subplot(gs[1])
        bars2= axB.barh(range(nb2),cv2,color=bc2,edgecolor='white',alpha=0.88,height=0.75)
        axB.set_yticks(range(nb2)); axB.set_yticklabels([str(l)[:50] for l in cl2],fontsize=9)
        axB.set_xlim(0,110); axB.set_xlabel('Per-Class Recall (%)',fontsize=12)
        axB.set_title(f'B: Top {TOP_N2} best & Bottom {BOT_N2} worst recall\n'
            f'Mean recall = {recall_per.mean():.1f}%',fontsize=12)
        for bar,val in zip(bars2,cv2):
            if np.isnan(val): continue
            axB.text(val+1.5,bar.get_y()+bar.get_height()/2,f'{val:.1f}%',va='center',fontsize=8.5)
        axB.legend(handles=[
            mpatches.Patch(color=PALETTE[1],alpha=0.85,label=f'Top {TOP_N2} recall'),
            mpatches.Patch(color=PALETTE[4],alpha=0.85,label=f'Bottom {BOT_N2} recall'),
        ],fontsize=10,loc='lower right')
        fig.suptitle(f'{name} — Normalised Confusion Matrix\n'
            f'Accuracy={acc:.4f}  F1={f1:.4f} | {N_CLASSES} classes | n_test={len(y_test)}',
            fontsize=13, y=1.01)

    plt.tight_layout()
    savefig(fname)
    if MODELS[name]['scaled']: del Xte
    del cm, cm_pct, y_pred
    gc.collect()

print('✓ Cell 10 complete')

  ✓  fig05_random_forest_confusion_matrix.png
  ✓  fig06_decision_tree_confusion_matrix.png
  ✓  fig07_logistic_regression_confusion_matrix.png
  ✓  fig08_lightgbm_confusion_matrix.png
✓ Cell 10 complete


---
## Cell 11 — Figure 09: Model Performance Bar Chart

In [12]:
metrics  = ['Accuracy','Balanced Acc','Precision','Recall','F1','MCC','Kappa','AUC']
# Only plot metrics that exist in results
metrics  = [m for m in metrics if m in results_df.columns]
x_labels = metrics
n_models = len(results_df)
x        = np.arange(len(metrics))
w        = 0.75/n_models

fig, ax  = plt.subplots(figsize=(16, 7))
for i,(name,color) in enumerate(zip(results_df.index,PALETTE)):
    vals   = [results_df.loc[name,m] if not pd.isna(results_df.loc[name,m]) else 0 for m in metrics]
    offset = (i - n_models/2 + 0.5)*w
    bars   = ax.bar(x+offset, vals, w*0.92,
                    label=name, color=color, alpha=0.88, edgecolor='white')
    for bar,v in zip(bars,vals):
        if v == 0: continue
        if v > 0.15:
            ax.text(bar.get_x()+bar.get_width()/2, v-0.04,
                    f'{v:.3f}', ha='center', va='top', fontsize=7.5, color='white', fontweight='bold')
        else:
            ax.text(bar.get_x()+bar.get_width()/2, v+0.012,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x); ax.set_xticklabels(x_labels, fontsize=10)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.12)
ax.axhline(0.85, color='gray', linestyle='--', lw=1.0, label='Target (0.85)')
ax.set_title(
    'Algorithm Performance Comparison — Held-Out Test Set\n'
    f'({N_CLASSES} disease classes; macro-averaged; n_test={len(y_test)})',
    pad=12, fontsize=13)
ax.legend(loc='upper right', fontsize=9, ncol=3)
plt.tight_layout()
savefig('fig09_model_comparison_bar.png')
print('✓ Cell 11 complete')

  ✓  fig09_model_comparison_bar.png
✓ Cell 11 complete


---
## Cell 12 — Figure 10: ROC Curves

In [13]:
fpr_grid = np.linspace(0,1,500)
fig, ax  = plt.subplots(figsize=(9,7))

for name,color in zip(results_df.index, PALETTE):
    if name not in y_probas: continue
    proba = y_probas[name]
    tprs  = []
    for k in np.unique(y_test):
        if (y_test==k).sum() < 2: continue
        y_bin_k = (y_test==k).astype(int)
        fpr_k,tpr_k,_ = roc_curve(y_bin_k, proba[:,k])
        tprs.append(np.interp(fpr_grid, fpr_k, tpr_k))
    if not tprs: continue
    mean_tpr = np.mean(tprs,axis=0); mean_tpr[0] = 0.0
    auc_val  = results_df.loc[name,'AUC']
    auc_str  = f'{auc_val:.4f}' if not pd.isna(auc_val) else 'N/A'
    ax.plot(fpr_grid, mean_tpr, color=color, lw=2.2, label=f'{name}  (AUC={auc_str})')
    del tprs, mean_tpr

ax.plot([0,1],[0,1],'k--',lw=1,label='Random (AUC=0.50)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(
    'Macro-Averaged ROC Curves — All Four Classifiers\n'
    f'(One-vs-Rest; {N_CLASSES} classes; n_test={len(y_test)})',
    pad=12, fontsize=13)
ax.legend(loc='lower right',fontsize=10)
ax.set_xlim(-0.01,1.01); ax.set_ylim(-0.01,1.02)
plt.tight_layout()
savefig('fig10_roc_curves.png')

# Free large proba arrays — no longer needed after this cell
del y_probas; gc.collect()
print('✓ Cell 12 complete')

  ✓  fig10_roc_curves.png
✓ Cell 12 complete


---
## Cell 13 — Figure 11: RF Feature Importances

In [14]:
rf_model    = trained['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=SYMPTOM_COLS)
top30_imp   = importances.sort_values(ascending=False).head(30)
labels_imp  = [s.replace('_',' ') for s in top30_imp.index[::-1]]

fig, ax = plt.subplots(figsize=(13,11))
colors_imp = plt.cm.plasma(np.linspace(0.85,0.15,30))
bars = ax.barh(range(30), top30_imp.values[::-1], color=colors_imp, edgecolor='white', height=0.72)
ax.set_yticks(range(30)); ax.set_yticklabels(labels_imp, fontsize=10)
for bar,val in zip(bars, top30_imp.values[::-1]):
    ax.text(val+top30_imp.max()*0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.5f}', va='center', fontsize=8.5)
ax.set_xlabel('Mean Decrease in Gini Impurity (Feature Importance)')
ax.set_title(
    'Random Forest — Top 30 Symptom Feature Importances\n'
    f'(Mean Decrease Gini; {rf_model.n_estimators} trees)',
    pad=12, fontsize=13)
plt.tight_layout()
savefig('fig11_feature_importance_rf.png')
print('✓ Cell 13 complete')

  ✓  fig11_feature_importance_rf.png
✓ Cell 13 complete


---
## Cell 14 — SHAP Computation

In [24]:
rng_shap = np.random.default_rng(RANDOM_STATE)
idx_shap = rng_shap.choice(len(X_test), size=min(SHAP_N,len(X_test)), replace=False)
X_shap   = X_test[idx_shap]
y_shap   = y_test[idx_shap]

print(f'Computing SHAP on {len(X_shap)} samples × {X_shap.shape[1]} features × {N_CLASSES} classes...')
explainer  = shap.TreeExplainer(rf_model)
shap_raw   = explainer.shap_values(X_shap)
shap_arr   = np.array(shap_raw, dtype=np.float32)  # (n_samples, n_features, n_classes)
print(f'SHAP array: {shap_arr.shape}  |  {shap_arr.nbytes/1024**2:.0f} MB')

mean_abs_shap = np.mean(np.abs(shap_arr), axis=(0,2))  # (n_features,)
y_pred_shap   = rf_model.predict(X_shap)
shap_pred     = np.stack(
    [shap_arr[i,:,y_pred_shap[i]] for i in range(len(X_shap))]
)
print('✓ Cell 14 complete — SHAP ready')

Computing SHAP on 150 samples × 377 features × 754 classes...
SHAP array: (150, 377, 754)  |  163 MB
✓ Cell 14 complete — SHAP ready


---
## Cell 15 — Figure 12: SHAP Beeswarm

In [25]:
TOP_N_SHAP  = 20
top20_idx   = np.argsort(mean_abs_shap)[::-1][:TOP_N_SHAP]
top20_names = [SYMPTOM_COLS[i].replace('_',' ') for i in top20_idx]

plt.figure(figsize=(12,9))
shap.summary_plot(shap_pred[:,top20_idx], X_shap[:,top20_idx],
    feature_names=top20_names, plot_type='dot',
    max_display=TOP_N_SHAP, show=False,
    color_bar_label='Feature value (1=present, 0=absent)')
plt.title(
    'SHAP Global Beeswarm — Top 20 Symptoms by Mean |SHAP| Value\n'
    f'(Random Forest; predicted-class SHAP; n={len(X_shap)} instances)\n'
    'Red=symptom present; Blue=absent; Right of zero=increases disease probability',
    pad=14, fontsize=11)
plt.tight_layout()
savefig('fig12_shap_summary_beeswarm.png')
print('✓ Cell 15 complete')

  ✓  fig12_shap_summary_beeswarm.png
✓ Cell 15 complete


---
## Cell 16 — Figure 13: SHAP Waterfall

In [26]:
proba_shap = rf_model.predict_proba(X_shap)
conf_shap  = proba_shap[np.arange(len(X_shap)), y_pred_shap]
correct    = y_pred_shap == y_shap
best_i     = (np.where(correct)[0][np.argmax(conf_shap[correct])]
              if correct.any() else np.argmax(conf_shap))

pred_cls_idx  = y_pred_shap[best_i]
pred_cls_name = CLASS_NAMES[pred_cls_idx]
true_cls_name = CLASS_NAMES[y_shap[best_i]]
confidence    = conf_shap[best_i]
sv_instance   = shap_arr[best_i, :, pred_cls_idx]

print(f'True: {true_cls_name}  |  Predicted: {pred_cls_name}  |  Confidence: {confidence:.2%}')

TOPK       = 15
top_idx    = np.argsort(np.abs(sv_instance))[::-1][:TOPK]
top_sv     = sv_instance[top_idx]
top_names  = [SYMPTOM_COLS[j].replace('_',' ') for j in top_idx]
top_feat   = X_shap[best_i, top_idx]
bar_colors = ['#D62728' if v>0 else '#1F77B4' for v in top_sv]

fig, ax = plt.subplots(figsize=(12,8))
bars = ax.barh(range(TOPK), top_sv[::-1], color=bar_colors[::-1], edgecolor='white', alpha=0.90, height=0.72)
ax.set_yticks(range(TOPK))
ax.set_yticklabels(
    [f'{n}  [{"✓" if int(v)==1 else "✗"}]' for n,v in zip(top_names[::-1],top_feat[::-1])],
    fontsize=10)
ax.axvline(0,color='black',lw=0.8)
for bar,val in zip(bars,top_sv[::-1]):
    offset = abs(top_sv).max()*0.012
    ax.text(val+(offset if val>=0 else -offset), bar.get_y()+bar.get_height()/2,
            f'{val:+.4f}', va='center', ha='left' if val>=0 else 'right', fontsize=9)
ax.legend(handles=[
    mpatches.Patch(color='#D62728',alpha=0.88,label='Increases disease probability'),
    mpatches.Patch(color='#1F77B4',alpha=0.88,label='Decreases disease probability'),
], loc='lower right', fontsize=10)
ax.set_xlabel('SHAP Value (impact on model output for predicted class)')
ax.set_title(
    f'SHAP Waterfall — Single Prediction Example\n'
    f'Predicted: {pred_cls_name}  |  True: {true_cls_name}  |  Confidence: {confidence:.2%}\n'
    '[✓] symptom present   [✗] symptom absent',
    pad=12, fontsize=11)
plt.tight_layout()
savefig('fig13_shap_waterfall_example.png')
print('✓ Cell 16 complete')

True: ear drum damage  |  Predicted: ear drum damage  |  Confidence: 43.98%
  ✓  fig13_shap_waterfall_example.png
✓ Cell 16 complete


---
## Cell 17 — Figure 14: SHAP Global Bar

In [27]:
mean_shap_df = pd.DataFrame({'symptom':SYMPTOM_COLS,'mean_shap':mean_abs_shap})\
               .sort_values('mean_shap',ascending=False).head(20)

raw_vals = mean_shap_df['mean_shap'].values[::-1]
max_val  = raw_vals.max()
if   max_val < 1e-3: scale, unit = 1e5, '×10⁻⁵'
elif max_val < 1e-2: scale, unit = 1e4, '×10⁻⁴'
elif max_val < 0.1:  scale, unit = 1e3, '×10⁻³'
else:                scale, unit = 1.0, ''

plot_vals  = raw_vals * scale
labels_bar = [s.replace('_',' ') for s in mean_shap_df['symptom'].values[::-1]]

fig, ax = plt.subplots(figsize=(13,9))
colors_g = plt.cm.plasma(np.linspace(0.85,0.15,20))
bars = ax.barh(range(20), plot_vals, color=colors_g, edgecolor='white', alpha=0.92, height=0.72)
ax.set_yticks(range(20)); ax.set_yticklabels(labels_bar, fontsize=10)
for bar,pv in zip(bars,plot_vals):
    ax.text(pv+plot_vals.max()*0.012, bar.get_y()+bar.get_height()/2,
            f'{pv:.2f}', va='center', fontsize=9)
unit_str = f' ({unit})' if unit else ''
ax.set_xlabel(f'Mean |SHAP Value|{unit_str}  —  averaged across samples and classes')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title(
    'SHAP Global Feature Importance — Top 20 Symptoms\n'
    f'(Random Forest; n={len(X_shap)} test instances)\n'
    'Higher bar = greater global influence on disease prediction',
    pad=12, fontsize=13)
plt.tight_layout()
savefig('fig14_shap_bar_global.png')

# Free SHAP arrays — large, no longer needed
del shap_arr, shap_pred, shap_raw
gc.collect()
print('✓ Cell 17 complete')

  ✓  fig14_shap_bar_global.png
✓ Cell 17 complete


---
## Cell 18 — Save Model Artifacts

In [28]:
def save_artifact(obj, filename):
    path = f'{ARTIFACT_DIR}/{filename}'
    with open(path,'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    kb = os.path.getsize(path)/1024
    print(f'  ✓  {filename:<38s} ({kb:.0f} KB)')

print('Saving artifacts...')
for name, mdl in trained.items():
    save_artifact(mdl, name.lower().replace(' ','_')+'.pkl')
save_artifact(trained[best_model_name], 'best_model.pkl')
if scaler is not None:
    save_artifact(scaler, 'scaler.pkl')
save_artifact(le, 'label_encoder.pkl')
pd.Series(SYMPTOM_COLS, name='symptom').to_csv(f'{ARTIFACT_DIR}/symptom_columns.csv', index=False)
print(f'  ✓  symptom_columns.csv  ({len(SYMPTOM_COLS)} symptoms)')
results_df.to_csv(f'{ARTIFACT_DIR}/model_performance.csv')
print(f'  ✓  model_performance.csv')
print(f'\nBest model : {best_model_name}')
print(f'  F1        = {results_df.loc[best_model_name,"F1"]:.4f}')
print(f'  MCC       = {results_df.loc[best_model_name,"MCC"]:.4f}')
print(f'  AUC       = {results_df.loc[best_model_name,"AUC"]:.4f}')
print(f'  Accuracy  = {results_df.loc[best_model_name,"Accuracy"]:.4f}')
needs_sc = best_model_name in ('Logistic Regression',)
print(f'  Needs scaler before predict(): {needs_sc}')
print('✓ Cell 18 complete')

Saving artifacts...
  ✓  random_forest.pkl                      (489197 KB)
  ✓  decision_tree.pkl                      (2882 KB)
  ✓  logistic_regression.pkl                (1120 KB)
  ✓  lightgbm.pkl                           (7775 KB)
  ✓  best_model.pkl                         (1120 KB)
  ✓  scaler.pkl                             (9 KB)
  ✓  label_encoder.pkl                      (16 KB)
  ✓  symptom_columns.csv  (377 symptoms)
  ✓  model_performance.csv

Best model : Logistic Regression
  F1        = 0.8430
  MCC       = 0.8683
  AUC       = 0.9998
  Accuracy  = 0.8687
  Needs scaler before predict(): True
✓ Cell 18 complete


---
## Cell 19 — Final Summary

In [29]:
print('═'*70)
print('  DISSERTATION NOTEBOOK — COMPLETE SUMMARY')
print('═'*70)
print('\n── Performance Table ────────────────────────────────────────────')
print(results_df.round(4).to_string())
print('\n── Cross-Validation ─────────────────────────────────────────────')
for name,scores in cv_scores.items():
    print(f'  {name:22s}: {scores.mean():.4f} ± {scores.std():.4f}')
print('\n── Figures ──────────────────────────────────────────────────────')
for f in sorted(os.listdir(FIG_DIR)):
    kb = os.path.getsize(f'{FIG_DIR}/{f}')/1024
    print(f'  {f:<50s} {kb:6.0f} KB')
print('\n── Artifacts ────────────────────────────────────────────────────')
for f in sorted(os.listdir(ARTIFACT_DIR)):
    kb = os.path.getsize(f'{ARTIFACT_DIR}/{f}')/1024
    print(f'  {f:<50s} {kb:6.0f} KB')
print(f'\n★  Best model: {best_model_name}')
print('═'*70)

══════════════════════════════════════════════════════════════════════
  DISSERTATION NOTEBOOK — COMPLETE SUMMARY
══════════════════════════════════════════════════════════════════════

── Performance Table ────────────────────────────────────────────
                     Accuracy  Balanced Acc  Precision  Recall      F1     MCC   Kappa     AUC
Logistic Regression    0.8687        0.8695     0.8384  0.8672  0.8430  0.8683  0.8683  0.9998
Random Forest          0.5460        0.6709     0.6740  0.6619  0.6082  0.5528  0.5449  0.9691
Decision Tree          0.0301        0.1043     0.1140  0.1038  0.0971  0.0779  0.0267     NaN
LightGBM               0.0062        0.0020     0.0012  0.0020  0.0008  0.0083  0.0013     NaN

── Cross-Validation ─────────────────────────────────────────────
  Random Forest         : 0.5788 ± 0.0113
  Decision Tree         : 0.0844 ± 0.0033
  Logistic Regression   : 0.8470 ± 0.0019
  LightGBM              : 0.0003 ± 0.0001

── Figures ──────────────────────────